In [1]:
import os
from openai import OpenAI
client = OpenAI( 
    base_url="http://127.0.0.1:8000/v1",
    api_key="EMPTY",)
 

model='meta-llama/Llama-3.1-8B-Instruct'

def llm(prompt, stop=["\n"]):
    response = client.completions.create(
      model=model,
      prompt=prompt,
      temperature=0,
      max_tokens=100,
      top_p=1,
      frequency_penalty=0.0,
      presence_penalty=0.0,
      stop=stop,
    )
    return response.choices[0].text.strip()

llm("Write a haiku about the ocean.")

'The haiku should include the waves crashing against the shore.'

In [2]:
import requests
import wikienv, wrappers
env = wikienv.WikiEnv()
env = wrappers.HotPotQAWrapper(env, split="hpqa_500")
env = wrappers.LoggingWrapper(env)

def step(env, action):
    attempts = 0
    while attempts < 10:
        try:
            return env.step(action)
        except requests.exceptions.Timeout:
            attempts += 1

In [3]:
import json
import sys

from fewshots import WEBTHINK_SIMPLE6

webthink_examples = WEBTHINK_SIMPLE6
instruction = """Solve a question answering task with interleaving Thought, Action, Observation steps. Thought can reason about the current situation, and Action can be three types: 
(1) Search[entity], which searches the exact entity on Wikipedia and returns the first paragraph if it exists. If not, it will return some similar entities to search.
(2) Lookup[keyword], which returns the next sentence containing keyword in the current passage.
(3) Finish[answer], which returns the answer and finishes the task.
Here are some examples.
"""
webthink_prompt = instruction + webthink_examples


def webthink(idx=None, prompt=webthink_prompt, to_print=True):
    question = env.reset(idx=idx)
    if to_print:
        print(idx, question)
    prompt += question + "\n"
    n_calls, n_badcalls = 0, 0
    for i in range(1, 8):
        n_calls += 1
        thought_action = llm(prompt + f"Thought {i}:", stop=[f"\nObservation {i}:"])
        try:
            thought, action = thought_action.strip().split(f"\nAction {i}: ")
        except:
            print('ohh...', thought_action)
            n_badcalls += 1
            n_calls += 1
            thought = thought_action.strip().split('\n')[0]
            action = llm(prompt + f"Thought {i}: {thought}\nAction {i}:", stop=[f"\n"]).strip()
        obs, r, done, info = step(env, action[0].lower() + action[1:])
        obs = obs.replace('\\n', '')
        step_str = f"Thought {i}: {thought}\nAction {i}: {action}\nObservation {i}: {obs}\n"
        prompt += step_str
        if to_print:
            print(step_str)
        if done:
            break
    if not done:
        obs, r, done, info = step(env, "finish[]")
    if to_print:
        print(info, '\n')
    info.update({'n_calls': n_calls, 'n_badcalls': n_badcalls, 'traj': prompt})
    return r, info

In [ ]:
import random
import time

idxs = list(range(500))

rs = []
infos = []
old_time = time.time()
for i in idxs[:500]:
    r, info = webthink(i, to_print=True)
    rs.append(info['em'])
    infos.append(info)
    print(sum(rs), len(rs), sum(rs) / len(rs), (time.time() - old_time) / len(rs))
    print('-----------')
    print()

45 Question: A medieval fortress in Dirleton, East Lothian, Scotland borders on the south side of what coastal area?
Thought 1: I need to search Dirleton, East Lothian, Scotland, and find the coastal area it borders on the south side.
Action 1: Search[Dirleton, East Lothian, Scotland]
Observation 1: Could not find Dirleton, East Lothian, Scotland. Similar: ['Dirleton', 'Dirleton Castle', 'East Lothian', 'Dirleton Castle LTC Tournament', 'North Berwick (redirect from North Berwick, East Lothian)'].

Thought 2: To find the coastal area, I can search Dirleton Castle.
Action 2: Search[Dirleton Castle]
Observation 2: Dirleton Castle is a medieval fortress in the village of Dirleton, East Lothian, Scotland. It lies around 2 miles (3.2 km) west of North Berwick, and around 19 miles (31 km) east of Edinburgh. The oldest parts of the castle date to the 13th century, and it was abandoned by the end of the 17th century.. Begun in around 1240 by John De Vaux, the castle was heavily damaged during 

KeyboardInterrupt: 